# Module 05 — Direct Preference Optimisation (DPO)

**Goal**: Train a model to prefer better responses using preference pairs — *without* a reward model or RL loop.

**What you will build**:
1. Format a preference dataset (chosen / rejected)
2. Train with `DPOTrainer`
3. Verify the model prefers the better response

In [ ]:
# Cell 1 — Install
!pip install -q transformers datasets trl peft accelerate torch

## 1. Why DPO?

RLHF with PPO (Module 04) requires:
- A separate reward model
- A value head
- A complex generate → score → update loop
- Many hyperparameters (KL coefficient, clip range, …)

**DPO** (Rafailov et al., 2023) shows that the same optimal policy can be
recovered *directly* from preference pairs, without an explicit reward model.

The key insight: the reward model is implicitly defined by the policy itself.

**DPO loss**:
```
L_DPO = -E[ log σ( β · log(π_θ(y_w|x)/π_ref(y_w|x))
                  - β · log(π_θ(y_l|x)/π_ref(y_l|x)) ) ]
```
- `y_w` = chosen response, `y_l` = rejected response
- `π_θ` = current policy, `π_ref` = frozen SFT reference
- `β` = temperature controlling how far we deviate from reference

In plain English: increase the probability of chosen responses *relative to the reference*,
and decrease the probability of rejected responses *relative to the reference*.

In [ ]:
# Cell 2 — Imports
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, TaskType

MODEL_ID  = "HuggingFaceTB/SmolLM2-135M-Instruct"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE     = torch.bfloat16
MAX_STEPS = 80
print(f"Device: {DEVICE}")

## 2. Preference dataset

Same format as Module 03: (prompt, chosen, rejected) triples.

In [ ]:
# Cell 3 — Dataset
PREFS = [
    {"prompt": "What is backpropagation?",
     "chosen": "Backpropagation is the algorithm used to compute gradients in a neural network. It applies the chain rule of calculus, starting from the loss and working backwards through each layer, computing how much each weight contributed to the error. These gradients are then used by an optimiser (e.g. SGD, Adam) to update the weights.",
     "rejected": "Backpropagation is when the network goes backwards. It computes gradients somehow."},
    {"prompt": "What is the softmax function?",
     "chosen": "Softmax converts a vector of raw scores (logits) into a probability distribution that sums to 1. For a vector z, softmax(z)_i = exp(z_i) / Σ exp(z_j). It amplifies the largest value and suppresses smaller ones, making the dominant class more certain.",
     "rejected": "Softmax is a math function. It makes numbers add up to 1."},
    {"prompt": "What is the purpose of dropout?",
     "chosen": "Dropout is a regularisation technique where, during training, each neuron is randomly set to zero with probability p (typically 0.1–0.5). This prevents co-adaptation — neurons can't rely on specific other neurons always being present — forcing the network to learn redundant representations and reducing overfitting.",
     "rejected": "Dropout randomly removes neurons during training to prevent overfitting."},
    {"prompt": "Explain the encoder-decoder architecture.",
     "chosen": "An encoder-decoder (seq2seq) model maps an input sequence to a fixed-size context (encoder), then generates an output sequence from that context (decoder). The encoder processes the full input and produces hidden representations; the decoder attends to those representations while autoregressively generating the output. Common applications include translation, summarisation, and code generation.",
     "rejected": "The encoder takes input and the decoder gives output. They are used in translation."},
    {"prompt": "What is knowledge distillation?",
     "chosen": "Knowledge distillation trains a small student model to mimic a large teacher model. Instead of using hard labels (0/1), the student is trained on the teacher's soft probability distributions, which contain richer information about the similarity between classes. This transfers the teacher's 'dark knowledge' into a compact model.",
     "rejected": "Knowledge distillation makes a small model learn from a big model."},
    {"prompt": "What is the role of the learning rate scheduler?",
     "chosen": "A learning rate scheduler adjusts the learning rate during training according to a predefined schedule. Common strategies include warmup (starting low, rising to the target rate) followed by cosine decay or linear decay. Warmup helps stabilise early training when gradients are noisy; decay prevents overshooting as the loss surface flattens.",
     "rejected": "A learning rate scheduler changes the learning rate. It's used to make training better."},
    {"prompt": "What is perplexity in language models?",
     "chosen": "Perplexity is a measure of how well a language model predicts a sample. It is defined as the exponent of the average negative log-likelihood per token: PPL = exp(-1/N · Σ log P(token_i)). Lower perplexity means the model is less 'surprised' by the text — it predicts it more confidently. A perplexity of k means the model is as uncertain as if it had to choose uniformly among k tokens at each step.",
     "rejected": "Perplexity measures how confused the model is. Lower is better."},
    {"prompt": "What is RLHF?",
     "chosen": "Reinforcement Learning from Human Feedback (RLHF) is a post-training technique that aligns a language model with human preferences. First, a reward model is trained on human preference comparisons. Then, the LM policy is optimised using PPO to maximise the reward signal while staying close to a reference policy via a KL penalty. This is how models like ChatGPT and Claude were made more helpful and safe.",
     "rejected": "RLHF uses human feedback to train models with reinforcement learning."},
] * 6  # 48 examples

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# DPOTrainer expects plain text columns "prompt", "chosen", "rejected"
pref_ds = Dataset.from_list(PREFS)
print(f"Dataset: {len(pref_ds)} preference pairs")
print("\nExample chosen response (truncated):")
print(pref_ds[0]["chosen"][:250])

## 3. Baseline inference (before DPO)

In [ ]:
# Cell 4 — Load model & baseline
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE
)

def infer(model, tok, question, max_new=120):
    msgs = [{"role": "user", "content": question}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_new,
                             temperature=0.7, do_sample=True,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

TEST = "What is perplexity in language models?"
print("BEFORE DPO:")
print(infer(model, tokenizer, TEST))

## 4. Train with DPOTrainer

In [ ]:
# Cell 5 — LoRA config (DPO with full fine-tuning is expensive; LoRA is efficient)
from peft import get_peft_model
lora_cfg = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# Cell 6 — DPOConfig
dpo_config = DPOConfig(
    output_dir="/tmp/dpo_model",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=8,
    beta=0.1,           # KL temperature — how much to stay near reference
    logging_steps=10,
    bf16=True,
    max_prompt_length=128,
    max_length=256,
    report_to="none",
)

In [ ]:
# Cell 7 — Train
dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=pref_ds,
    processing_class=tokenizer,
)

print("Starting DPO training...")
dpo_trainer.train()
print("Done!")

In [ ]:
# Cell 8 — After DPO
merged = model.merge_and_unload()
merged.eval()

print("AFTER DPO:")
print(infer(merged, tokenizer, TEST))

In [ ]:
# Cell 9 — DPO loss curve
import matplotlib.pyplot as plt

log = dpo_trainer.state.log_history
steps  = [x["step"] for x in log if "loss" in x]
losses = [x["loss"] for x in log if "loss" in x]
rewards_chosen   = [x.get("rewards/chosen",  0) for x in log if "loss" in x]
rewards_rejected = [x.get("rewards/rejected", 0) for x in log if "loss" in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(steps, losses, color="#2a9d8f", linewidth=2)
ax1.set_title("DPO Loss"); ax1.set_xlabel("Step"); ax1.grid(alpha=0.3)

ax2.plot(steps, rewards_chosen,   color="#264653", linewidth=2, label="Chosen")
ax2.plot(steps, rewards_rejected, color="#e9c46a", linewidth=2, label="Rejected")
ax2.set_title("Implicit Reward (chosen vs rejected)"); ax2.set_xlabel("Step")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/dpo_curves.png", dpi=120)
plt.show()
print("Chosen reward should be rising above rejected reward ↑")

## ✅ Summary

| | PPO (Module 04) | DPO (this module) |
|-|-----------------|-------------------|
| **Reward model** | Required | Not needed |
| **RL loop** | Required | Not needed |
| **Data** | Prompts only | Preference pairs |
| **Complexity** | High | Low |
| **Stability** | Can be unstable | Stable |
| **Performance** | Slightly better on hard tasks | Competitive on most tasks |

**Next**: Module 06 implements GRPO — which uses *verifiable rewards* (math correctness, code tests) instead of a learned reward model. This is the approach behind DeepSeek-R1 and similar reasoning models.

> 💡 DPO is the most practical first choice for alignment fine-tuning. Use PPO when you have a strong reward model and need to push performance on a specific dimension.